# Sustained Operation & Anti-Islanding Conformance
Used to calculate conformance results for two AS/NZS 4777.2:2020 (Australia-A, or otherwise) voltage-protection mechanisms 

Write them to *parallel review* Iceberg tables (`review_conformance_sust_op`, `review_conformance_antiisland`).
These tables **do not overwrite** the originals; once validated they can be promoted.
TBD -> Synthesise all results and remove reviews


### Tests performed:


#### Anti-islanding (AS/NZS 4777.2 Section 4.2., Table 4.1)
| Level | Threshold | Max disconnect time |
|---|---|---|
| UV2 | < 70 V | 2 s |
| UV1 | < 180 V | 11 s |
| OV1 | > 265 V | 2 s |
| OV2 | > 275 V | 0.2 s |

(note on trip delay time: Cannot be verified with a temporal resolution of 5-min per timestamp)

#### Sustained operation (AS/NZS 4777.2 Section 4.3., Table 4.3)
> "The inverter shall operate the automatic disconnection device (see Clause 4.2) within 3 s when the average voltage for a 10 min period exceeds the V_nom-max specified in Table 4.3."

With 5-min telemetry the closest faithful approximation of a "10-minute average" is:

```
V_10min_rolling[t] = (V[t] + V[t-1]) / 2     # rolling mean of current + previous interval
```

An inverter is **non-conforming at interval t** if:

1. `V_10min_rolling[t] > 258 V`  (the 10-min average test is triggered)
2. The inverter is *still generating*: `P_kW[t] > 0.04 * S_rated` (same 4%-nameplate floor as in Volt-Watt / Volt-VAr. Consistent across other modes and protective functions)

Non-conformance **magnitude** = `max(0, P_kW[t] − 0.04 * S_rated)`.

A rolling average can trigger while one of the two individual readings is just below the threshold.

**[!] Temporal resolution limitation.** Both thresholds require sub-second disconnection.
Our telemetry is 5-minute averages. We **cannot verify trip timing** at this resolution.

What we *can* observe: a 5-min interval where `V ≥ threshold` (i.e. voltage was at or above the threshold for some portion of those 5 minutes) AND the site was still generating.  
We flag this as a **potential non-conformance**, with the honest label that we are detecting "sustained over-voltage failure to trip".
The same physical failure as sustained operation, just in the higher OV1/OV2 band. 
We apply the same rolling-average framework for consistency.

The sweep over multiple `v_threshold` values (260~275 V) is retained so downstream analysis can choose the canonical level for reporting.

#### What both tests share
- `V = max(voltage)` across circuits at the site level (worst-phase view).
- 4%-nameplate floor for "still generating": `P_kW > 0.04 * ac_capacity_kw`.
- Non-conformance magnitude = `max(0, P_kW − 0.04 * ac_capacity_kw)`.
- `total_count` = intervals where the rolling-average test was triggered (the denominator   for the non-conforming fraction used by the 10%-rule site-level test).
- Results stored at daily + site grain with `v_threshold` as a dimension column.


## Environment setup

In [1]:
import boto3
import awswrangler as wr
import pandas as pd
import numpy as np
from time import sleep

# AWS auth
# Kernels launched from VSCode don't inherit the shell's AWS_PROFILE.
# Set it explicitly here.  
# Change profile name if project differs:
session = boto3.Session(profile_name="ciccada", region_name="ap-southeast-2")
wr.config.boto3_session = session

# Catalog shortcuts
SAI = "solar_analytics_iceberg"   # Iceberg. This is where results are stored.

# AS/NZS 4777.2 constants
# TBD -> Generalise and move to a constants module shared across the project.
AS4777 = {
    # 4%-nameplate floor: inverter is "generating" if P > this fraction of S_rated.
    # Consistent with the tolerance band used in V-Watt / V-VAr / inherited tables.
    "TOL_FRAC":         0.04,
    # Site conformant if non-conforming fraction of evaluated intervals <= 10%.
    "SITE_CONF_THRESH": 0.10,
    # Interval length in hours (for kW·h conversions if needed later)
    "INTERVAL_H":       5 / 60,
    # Sustained operation: 10-min rolling average must NOT exceed this value.
    # Source: AS/NZS 4777.2:2020 clause 3.3.3.
    "SUSTOP_V_CEIL":    258.0,
    # Anti-islanding OV thresholds (Table 3.4).
    # We sweep over these; the analyst picks the canonical one for reporting.
    "AI_OV1":           265.0,   # max disconnect 2 s
    "AI_OV2":           275.0,   # max disconnect 0.2 s
}

print("Session region:", session.region_name)
print("Constants:", AS4777)

Session region: ap-southeast-2
Constants: {'TOL_FRAC': 0.04, 'SITE_CONF_THRESH': 0.1, 'INTERVAL_H': 0.08333333333333333, 'SUSTOP_V_CEIL': 258.0, 'AI_OV1': 265.0, 'AI_OV2': 275.0}


## 1. Athena query wrapper

We use `awswrangler.athena.read_sql_query` rather than the `aq()` helper in `aws_config.py` so this notebook is self-contained and can be run standalone.

Future version can refer back to aq(), just need to swap it in. the SQL is identical.


In [10]:
import awswrangler as wr

database = SAI  # or database = "your_database_name"

def aq(sql, database=database, **kwargs):
    return wr.athena.read_sql_query(
        sql=sql,
        database=database,
        boto3_session=session,
        ctas_approach=False,
        **kwargs
    )

def aq_exec(sql, database=database, **kwargs):
    qid = wr.athena.start_query_execution(
        sql=sql,
        database=database,
        boto3_session=session,
        wait=True,
        **kwargs
    )
    print("Executed:", qid)
    return qid

# Quick connectivity check
try:
    ping = aq("SELECT 1 AS ok", database=database)
    print("Athena connection OK:", ping.to_dict("records"))
except Exception as e:
    print(f"Athena connection FAILED: {e}")




c:\Users\z3553082\AppData\Local\miniforge3\envs\ciccada\Lib\site-packages\awswrangler\athena\_read.py:602: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


Athena connection OK: [{'ok': 1}]


In [18]:
# Figure out the S3 prefix:
glue = session.client("glue")
db = glue.get_database(Name=SAI)["Database"]
print(db.get("LocationUri"))

s3://project-ciccada/Trino-Warehouse/solar_analytics/


## 2. Create the two *review* result tables

We write to `review_conformance_sust_op` and `review_conformance_antiisland`
so we **never touch** the original `conformance_sust_op_3w` / `conformance_antiisland`.

The schema deliberately mirrors the originals (same column names, same grain)
so the analysis notebook (`02_conformance_curtailment_analysis.ipynb`) can be pointed
at these tables with a one-line swap once they are validated.

The only structural change is an added `method` column (varchar) so you can store
both the rolling-average approach (this notebook) and the inherited approach in the same
table if you ever want a direct comparison.


In [20]:
# Pick the correct S3 prefix for Iceberg tables.
# Use the same bucket/prefix pattern as your other tables in solar_analytics_iceberg.
table = "review_conformance_sust_op"
base_location = db["LocationUri"].rstrip("/")
table_location = f"{base_location}/{table}/"

aq_exec(f"DROP TABLE IF EXISTS {table}")

aq_exec(f"""
CREATE TABLE {table} (
    year                         INT,
    month                        INT,
    day                          INT,
    day_night                    STRING,
    site_id                      BIGINT,
    method                       STRING,
    v_threshold                  DOUBLE,
    nonconformance_sust_op_sum   DOUBLE,
    nonconformance_sust_op_count BIGINT,
    total_count                  BIGINT
)
LOCATION '{table_location}'
TBLPROPERTIES (
    'table_type'='ICEBERG',
    'format'='parquet'
)
""")

Executed: {'QueryExecutionId': 'd4fd5125-01ee-4cbb-9710-1de446c52318', 'Query': 'DROP TABLE IF EXISTS review_conformance_sust_op', 'StatementType': 'DDL', 'ResultConfiguration': {'OutputLocation': 's3://aws-athena-query-results-130340360668-ap-southeast-2/d4fd5125-01ee-4cbb-9710-1de446c52318.txt'}, 'ResultReuseConfiguration': {'ResultReuseByAgeConfiguration': {'Enabled': False}}, 'QueryExecutionContext': {'Database': 'solar_analytics_iceberg'}, 'Status': {'State': 'SUCCEEDED', 'SubmissionDateTime': datetime.datetime(2026, 7, 10, 12, 3, 45, 872000, tzinfo=tzlocal()), 'CompletionDateTime': datetime.datetime(2026, 7, 10, 12, 3, 46, 179000, tzinfo=tzlocal())}, 'Statistics': {'EngineExecutionTimeInMillis': 139, 'DataScannedInBytes': 0, 'TotalExecutionTimeInMillis': 307, 'QueryQueueTimeInMillis': 133, 'ServicePreProcessingTimeInMillis': 15, 'ServiceProcessingTimeInMillis': 20, 'ResultReuseInformation': {'ReusedPreviousResult': False}}, 'WorkGroup': 'primary', 'EngineVersion': {'SelectedEngin

{'QueryExecutionId': 'e0052d8d-3281-4f19-af4e-05332ce48df3',
 'Query': "CREATE TABLE review_conformance_sust_op (\n    year                         INT,\n    month                        INT,\n    day                          INT,\n    day_night                    STRING,\n    site_id                      BIGINT,\n    method                       STRING,\n    v_threshold                  DOUBLE,\n    nonconformance_sust_op_sum   DOUBLE,\n    nonconformance_sust_op_count BIGINT,\n    total_count                  BIGINT\n)\nLOCATION 's3://project-ciccada/Trino-Warehouse/solar_analytics/review_conformance_sust_op/'\nTBLPROPERTIES (\n    'table_type'='ICEBERG',\n    'format'='parquet'\n)",
 'StatementType': 'DDL',
 'ResultConfiguration': {'OutputLocation': 's3://aws-athena-query-results-130340360668-ap-southeast-2/e0052d8d-3281-4f19-af4e-05332ce48df3.txt'},
 'ResultReuseConfiguration': {'ResultReuseByAgeConfiguration': {'Enabled': False}},
 'QueryExecutionContext': {'Database': 'solar_anal

In [8]:
# Diagnostic cell — run this to check required names/functions
required = ["AS4777", "SAI", "database", "session", "ping", "sleep", "aq", "aq_exec", "wr", "boto3"]
missing = [name for name in required if name not in globals()]
if missing:
    print("Missing definitions:", missing)
    print("Suggested fixes:")
    print("- Re-run the notebook cells that define the missing names (environment/athena helper cells).")
    print("- If that fails, Kernel -> Restart & Run All.")
else:
    print("All required names present.")
    print("AS4777 keys:", list(AS4777.keys()))
    print("SAI:", SAI)
    print("database:", database)
    print("session.region:", getattr(session, "region_name", None))
    print("ping:")
    display(ping.head())

All required names present.
AS4777 keys: ['TOL_FRAC', 'SITE_CONF_THRESH', 'INTERVAL_H', 'SUSTOP_V_CEIL', 'AI_OV1', 'AI_OV2']
SAI: solar_analytics_iceberg
database: solar_analytics_iceberg
session.region: ap-southeast-2
ping:


,ok
0,1


All required names present.
AS4777 keys: ['TOL_FRAC', 'SITE_CONF_THRESH', 'INTERVAL_H', 'SUSTOP_V_CEIL', 'AI_OV1', 'AI_OV2']
SAI: solar_analytics_iceberg
database: solar_analytics_iceberg
session.region: ap-southeast-2
ping:


,ok
0,1


In [ ]:
# ── Anti-islanding review table ───────────────────────────────────────────────
aq_exec("DROP TABLE IF EXISTS review_conformance_antiisland")
aq_exec("""
    CREATE TABLE review_conformance_antiisland (
        year                              INT,
        month                             INT,
        day                               INT,
        day_night                         VARCHAR,
        site_id                           BIGINT,
        method                            VARCHAR,
        v_threshold                       DOUBLE,
        nonconformance_antiisland_sum     DOUBLE,
        nonconformance_antiisland_count   BIGINT,
        total_count                       BIGINT
    )
""")
print("review_conformance_antiisland created.")


## 3. Core SQL template — rolling-average approach

Both sustained-op and anti-islanding share the same SQL skeleton.  
The parameterised variables are:

| Variable | Sustained op | Anti-islanding |
|---|---|---|
| `v_threshold` | 258 (standard) | 265 or 275 (swept) |
| `tol_frac` | 0.04 | 0.04 |
| `target_table` | `review_conformance_sust_op` | `review_conformance_antiisland` |
| `method_label` | `'rolling_2interval'` | `'rolling_2interval'` |
| `nc_col_name` | `nonconformance_sust_op_sum` | `nonconformance_antiisland_sum` |

The key CTE steps:

```
raw_data   — collapse circuit → site: P_kW, V = MAX(voltage), S_rated
with_lag   — add lag(V) and lag(P_kW) using window functions partitioned by site_id
rolling    — compute V_rolling = (V + lag_V) / 2  (≈ 10-min average)
flagged    — keep rows where V_rolling > v_threshold AND inverter near-generating
             nonconformance = max(0, P_kW - tol_frac * S_rated)
agg        — group to (year, month, day, day_night, site_id)
```

> **Why `MAX(voltage)` across circuits?**  
> Sustained-op / anti-islanding are protection functions — we want the worst-phase voltage,
> consistent with the inherited approach and with the fact that a single phase exceeding the
> limit is enough to require a trip.  Contrast with Volt-Watt / Volt-VAr which use `AVG`.

> **Why `voltage < 350`?**  
> We are testing the OV1 (≥265 V) and OV2 (≥275 V) bands, so we must not clip at 300 V
> (which is what Volt-Watt / Volt-VAr do).  350 V is a generous sanity bound to exclude
> obviously corrupt readings.


In [ ]:
def build_insert_sql(
    year, month, v_threshold, postcode_filter,
    target_table, nc_sum_col, nc_count_col, method_label,
    tol_frac=AS4777["TOL_FRAC"],
):
    """
    Build the INSERT ... SELECT SQL for one (year, month, v_threshold, postcode_shard).

    Parameters
    ----------
    year, month        : int  — partition to process
    v_threshold        : float — voltage threshold (e.g. 258.0, 265.0, 275.0)
    postcode_filter    : str  — Athena bucket predicate for cost control,
                                e.g. "system.bucket(postcode, 4) = 0"
    target_table       : str  — Iceberg table to INSERT into
    nc_sum_col         : str  — column name for the kW-excess sum
    nc_count_col       : str  — column name for the NC interval count
    method_label       : str  — stored in the `method` column for traceability
    tol_frac           : float — 4%-nameplate generating floor (default from AS4777)
    """
    return f"""
INSERT INTO {target_table}
WITH

-- ── Step 1: collapse circuits → site×timestamp ────────────────────────────────
-- P_kW  = net export, kW (positive = exporting)
-- V     = MAX(voltage) across all PV circuits at the site (worst-phase view)
-- S_rated = ac_capacity_kw (nameplate; from meta_up23c)
raw_data AS (
    SELECT
        m.site_id,
        ts.t_stamp,
        SUM(CAST(ts.power * m.circuit_polarity AS DECIMAL(18,6))) / 1000.0 AS P_kW,
        MAX(CAST(ts.voltage AS DECIMAL(18,6)))                              AS V,
        MAX(m.ac_capacity_kw)                                               AS S_rated
    FROM
        ts
        INNER JOIN (
            SELECT site_id, circuit_id, circuit_polarity, ac_capacity_kw
            FROM   meta_up23c
            -- no is_pv filter here because we join on circuit_id which is already PV
        ) AS m ON ts.circuit_id = m.circuit_id
    WHERE
        ts.year     = {year}
        AND ts.month    = {month}
        AND ts.is_pv    = TRUE
        AND ts.voltage  > 0
        AND ts.voltage  < 350          -- wide upper bound: we test >=265/275 V here
        AND {postcode_filter}          -- cost-control shard
    GROUP BY m.site_id, ts.t_stamp
),

-- ── Step 2: add one-interval lag for V and P ──────────────────────────────────
-- We need lag(V) to compute the 2-interval rolling average and lag(P_kW)
-- to check whether the inverter was generating in the PRIOR interval
-- (helps catch the case where it tripped exactly at the boundary).
with_lag AS (
    SELECT
        site_id,
        t_stamp,
        P_kW,
        V,
        S_rated,
        LAG(t_stamp) OVER (PARTITION BY site_id ORDER BY t_stamp) AS prev_t_stamp,
        LAG(V)       OVER (PARTITION BY site_id ORDER BY t_stamp) AS prev_V,
        LAG(P_kW)    OVER (PARTITION BY site_id ORDER BY t_stamp) AS prev_P_kW
    FROM raw_data
),

-- ── Step 3: rolling 2-interval average of voltage ────────────────────────────
-- V_rolling ≈ 10-minute average voltage (two consecutive 5-min readings).
-- We only compute this when the previous reading is exactly 5 minutes earlier
-- (i.e., no data gap).  If there is a gap, prev_V is NULL and V_rolling will
-- be NULL, which means the row is NOT flagged — conservative / correct.
rolling AS (
    SELECT
        site_id,
        t_stamp,
        P_kW,
        V,
        S_rated,
        prev_t_stamp,
        prev_V,
        prev_P_kW,
        -- 10-min rolling average: only valid when readings are consecutive.
        CASE
            WHEN prev_t_stamp IS NOT NULL
             AND t_stamp - prev_t_stamp = INTERVAL '5' MINUTE
            THEN (V + prev_V) / 2.0
            ELSE NULL
        END AS V_rolling,
        -- day/night flag derived from UTC t_stamp.
        -- 'day' = hour(UTC) in [20,23] or [0,7] ≈ 06:00-17:00 AEST.
        -- Consistent with all other conformance tables in this project.
        CASE
            WHEN hour(t_stamp) >= 20 OR hour(t_stamp) <= 7 THEN 'day'
            ELSE 'night'
        END AS day_night
    FROM with_lag
),

-- ── Step 4: flag non-conforming intervals ─────────────────────────────────────
-- Eligibility condition (what enters total_count):
--   V_rolling > v_threshold    (10-min average exceeds the limit)
--   AND the inverter is currently or was recently generating
--
-- "Currently or recently generating" = P_kW > tol OR prev_P_kW > tol
--   The prev_P_kW arm catches the interval immediately after a trip: if the
--   inverter was above 4%-nameplate last interval and now it has tripped (P=0),
--   we still want that boundary interval in the denominator so the fraction is
--   meaningful.  This mirrors the inherited approach.
--
-- Non-conformance magnitude = excess above the 4%-floor.
--   If the inverter has actually tripped (P_kW <= 4%-floor), nc = 0.
--   We count that interval in total_count but not in nc_count — correct behaviour.
flagged AS (
    SELECT
        site_id,
        t_stamp,
        V_rolling,
        V,
        P_kW,
        S_rated,
        day_night,
        day(t_stamp)   AS day,
        month(t_stamp) AS month,
        year(t_stamp)  AS year,
        GREATEST(0.0, P_kW - {tol_frac} * S_rated) AS nc_kW
    FROM rolling
    WHERE
        V_rolling > {v_threshold}                         -- rolling average test triggered
        AND (
               P_kW     > {tol_frac} * S_rated            -- inverter generating now
            OR prev_P_kW > {tol_frac} * S_rated           -- or was generating last interval
        )
),

-- ── Step 5: aggregate to (year, month, day, day_night, site_id) ───────────────
agg AS (
    SELECT
        year,
        month,
        day,
        day_night,
        site_id,
        '{method_label}'   AS method,
        {v_threshold}      AS v_threshold,
        SUM(nc_kW)                                        AS {nc_sum_col},
        SUM(CASE WHEN nc_kW > 0 THEN 1 ELSE 0 END)        AS {nc_count_col},
        COUNT(nc_kW)                                      AS total_count
    FROM flagged
    GROUP BY year, month, day, day_night, site_id
)

SELECT * FROM agg
"""


## 4. Run: Sustained operation (v_threshold = 258 V)

We run for **2024 and 2025** to match the inherited table coverage.

The postcode-bucket sharding controls Athena scan cost: splitting into 4 buckets means
each query scans roughly ¼ of the data.  Increase `n_buckets` if queries time out.

> **Why only 258 V for sustained-op?**  
> The standard is explicit: 258 V is the 10-minute-average upper limit.  
> If you want a sensitivity sweep (e.g. to match the inherited 253–258 range),
> add thresholds to `SUST_OP_THRESHOLDS` — but the canonical reporting value is 258.


In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
SUST_OP_THRESHOLDS = [258.0]     # canonical; add more for sensitivity, e.g. [253,255,258]
YEARS              = [2024, 2025]
MONTHS             = list(range(1, 13))
N_BUCKETS          = 4            # postcode shards (increase if queries are slow/timeout)

SUST_OP_TABLE   = "review_conformance_sust_op"
SUST_OP_NC_SUM  = "nonconformance_sust_op_sum"
SUST_OP_NC_CNT  = "nonconformance_sust_op_count"
SUST_OP_METHOD  = "rolling_2interval"

# ── Run ───────────────────────────────────────────────────────────────────────
errors_sustop = []

for year in YEARS:
    for month in MONTHS:
        for v_thr in SUST_OP_THRESHOLDS:
            for bucket in range(N_BUCKETS):
                postcode_filter = (
                    f"system.bucket(postcode, {N_BUCKETS}) = {bucket}"
                )
                label = f"sust_op | year={year} m={month:02d} v={v_thr} shard={bucket}"
                sql = build_insert_sql(
                    year          = year,
                    month         = month,
                    v_threshold   = v_thr,
                    postcode_filter = postcode_filter,
                    target_table  = SUST_OP_TABLE,
                    nc_sum_col    = SUST_OP_NC_SUM,
                    nc_count_col  = SUST_OP_NC_CNT,
                    method_label  = SUST_OP_METHOD,
                )
                try:
                    aq_exec(sql)
                    print(f"  ✓ {label}")
                except Exception as e:
                    print(f"  ✗ {label}: {e}")
                    errors_sustop.append((label, str(e)))
                sleep(2)   # brief pause between inserts

print(f"\nSustained-op done. Errors: {len(errors_sustop)}")
if errors_sustop:
    for lbl, err in errors_sustop:
        print(f"  {lbl}: {err}")


## 5. Run: Anti-islanding (OV1 = 265 V, OV2 = 275 V sweep)

We sweep over both OV thresholds and store `v_threshold` as a column.
The analysis notebook can then select the canonical level.

> **Limitation note (copy into any report):**  
> We cannot verify the 2-second / 0.2-second trip timing requirement at 5-min resolution.
> A flagged interval means: in a 5-minute window where the site-level maximum voltage
> (worst phase) rolling average was ≥ `v_threshold`, the inverter continued to export
> more than 4% of nameplate.  This is a necessary but not sufficient condition for a
> trip-timing violation.  Events where an inverter correctly tripped within 2 s but
> voltage remained elevated for the rest of the 5-min window are indistinguishable from
> genuine failures at this resolution.


In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
AI_THRESHOLDS = [265.0, 275.0]   # OV1 and OV2; add 260.0 for a lower-bound sensitivity
YEARS         = [2024, 2025]
MONTHS        = list(range(1, 13))
N_BUCKETS     = 4

AI_TABLE      = "review_conformance_antiisland"
AI_NC_SUM     = "nonconformance_antiisland_sum"
AI_NC_CNT     = "nonconformance_antiisland_count"
AI_METHOD     = "rolling_2interval"

# ── Run ───────────────────────────────────────────────────────────────────────
errors_ai = []

for year in YEARS:
    for month in MONTHS:
        for v_thr in AI_THRESHOLDS:
            for bucket in range(N_BUCKETS):
                postcode_filter = (
                    f"system.bucket(postcode, {N_BUCKETS}) = {bucket}"
                )
                label = f"antiisland | year={year} m={month:02d} v={v_thr} shard={bucket}"
                sql = build_insert_sql(
                    year            = year,
                    month           = month,
                    v_threshold     = v_thr,
                    postcode_filter = postcode_filter,
                    target_table    = AI_TABLE,
                    nc_sum_col      = AI_NC_SUM,
                    nc_count_col    = AI_NC_CNT,
                    method_label    = AI_METHOD,
                )
                try:
                    aq_exec(sql)
                    print(f"  ✓ {label}")
                except Exception as e:
                    print(f"  ✗ {label}: {e}")
                    errors_ai.append((label, str(e)))
                sleep(2)

print(f"\nAnti-islanding done. Errors: {len(errors_ai)}")
if errors_ai:
    for lbl, err in errors_ai:
        print(f"  {lbl}: {err}")


## 6. Sanity checks

Quick row counts and site counts to confirm the tables populated and look reasonable.
Compare with the inherited tables if available.


In [ ]:
# ── Row / site counts per table, year, v_threshold ────────────────────────────
for tbl, name in [
    ("review_conformance_sust_op",   "Sustained-op (review)"),
    ("review_conformance_antiisland", "Anti-islanding (review)"),
]:
    try:
        df = aq(f"""
            SELECT year, v_threshold,
                   COUNT(*)                AS n_rows,
                   COUNT(DISTINCT site_id) AS n_sites,
                   SUM(total_count)        AS total_intervals_evaluated,
                   SUM(nonconformance_sust_op_count
                       + 0)                AS nc_count_sum   -- placeholder; fixed below
            FROM {tbl}
            GROUP BY year, v_threshold
            ORDER BY year, v_threshold
        """)
        print(f"\n=== {name} ({tbl}) ===")
        display(df)
    except Exception as e:
        print(f"{tbl}: {e}")


In [ ]:
# ── Re-run with correct NC column names ──────────────────────────────────────
for tbl, nc_col in [
    ("review_conformance_sust_op",    "nonconformance_sust_op_count"),
    ("review_conformance_antiisland", "nonconformance_antiisland_count"),
]:
    try:
        df = aq(f"""
            SELECT
                year,
                v_threshold,
                COUNT(*)                AS n_rows,
                COUNT(DISTINCT site_id) AS n_sites,
                SUM(total_count)        AS total_intervals_evaluated,
                SUM({nc_col})           AS nc_intervals_total,
                ROUND(
                    100.0 * SUM({nc_col}) / NULLIF(SUM(total_count), 0),
                    2
                )                       AS fleet_pct_nonconforming
            FROM {tbl}
            GROUP BY year, v_threshold
            ORDER BY year, v_threshold
        """)
        print(f"\n=== {tbl} ===")
        display(df)
    except Exception as e:
        print(f"ERROR {tbl}: {e}")


In [ ]:
# ── Compare site counts with inherited tables (if they exist) ─────────────────
# Comment out if the original tables were dropped.
for orig, rev, v_thr in [
    ("conformance_sust_op_3w",  "review_conformance_sust_op",    258),
    ("conformance_antiisland",  "review_conformance_antiisland", 265),
]:
    try:
        n_orig = aq(f"""
            SELECT COUNT(DISTINCT site_id) AS n
            FROM {orig} WHERE v_threshold = {v_thr}
        """)["n"].iloc[0]
        n_rev = aq(f"""
            SELECT COUNT(DISTINCT site_id) AS n
            FROM {rev} WHERE v_threshold = {v_thr}
        """)["n"].iloc[0]
        print(f"{orig:35s} sites @ {v_thr}V: {n_orig:>7,}")
        print(f"{rev:35s} sites @ {v_thr}V: {n_rev:>7,}")
        print()
    except Exception as e:
        print(f"Comparison skipped ({orig}): {e}")


## 7. Single-site verification

Pull raw 5-min data for one site-month and replicate the rolling-average logic in pandas
to confirm the SQL gives the same answer. Change `SITE_ID`, `YEAR`, `MONTH` as needed.


In [ ]:
SITE_ID = 625794481   # <── change to any site in the result tables
YEAR    = 2024
MONTH   = 10

# ── Pull raw telemetry for the site ──────────────────────────────────────────
raw = aq(f"""
    SELECT
        m.site_id,
        ts.t_stamp,
        SUM(CAST(ts.power * m.circuit_polarity AS DECIMAL(18,6))) / 1000.0 AS P_kW,
        MAX(CAST(ts.voltage AS DECIMAL(18,6)))                              AS V,
        MAX(m.ac_capacity_kw)                                               AS S_rated
    FROM ts
    INNER JOIN (
        SELECT site_id, circuit_id, circuit_polarity, ac_capacity_kw
        FROM   meta_up23c
    ) AS m ON ts.circuit_id = m.circuit_id
    WHERE ts.year   = {YEAR}
      AND ts.month  = {MONTH}
      AND ts.is_pv  = TRUE
      AND ts.voltage > 0
      AND ts.voltage < 350
      AND m.site_id = {SITE_ID}
    GROUP BY m.site_id, ts.t_stamp
    ORDER BY ts.t_stamp
""")

print(f"Rows fetched: {len(raw):,}")
display(raw.head())


In [ ]:
# ── Replicate the rolling-average + flagging logic in pandas ──────────────────
tol  = AS4777["TOL_FRAC"]
v258 = AS4777["SUSTOP_V_CEIL"]
v265 = AS4777["AI_OV1"]

df = raw.copy()
df["t_stamp"] = pd.to_datetime(df["t_stamp"], utc=True)
df = df.sort_values("t_stamp").reset_index(drop=True)

# lag columns
df["prev_t_stamp"] = df["t_stamp"].shift(1)
df["prev_V"]       = df["V"].shift(1)
df["prev_P_kW"]    = df["P_kW"].shift(1)

# rolling average: only valid when consecutive (5-min gap exactly)
gap = (df["t_stamp"] - df["prev_t_stamp"]).dt.total_seconds()
df["V_rolling"] = np.where(gap == 300, (df["V"] + df["prev_V"]) / 2.0, np.nan)

# day/night (UTC)
df["day_night"] = np.where(
    (df["t_stamp"].dt.hour >= 20) | (df["t_stamp"].dt.hour <= 7),
    "day", "night",
)

# generating flag
df["generating"]      = df["P_kW"]     > tol * df["S_rated"]
df["prev_generating"] = df["prev_P_kW"] > tol * df["S_rated"]

for thr, label in [(v258, f"{v258}V (sust-op)"), (v265, f"{v265}V (anti-island OV1)")]:
    eligible = (df["V_rolling"] > thr) & (df["generating"] | df["prev_generating"])
    flagged  = eligible.copy()
    df_e     = df[eligible].copy()
    df_e["nc_kW"] = np.maximum(0, df_e["P_kW"] - tol * df_e["S_rated"])
    total   = eligible.sum()
    nc_cnt  = (df_e["nc_kW"] > 0).sum()
    nc_sum  = df_e["nc_kW"].sum()
    print(f"  {label}: {total:4d} eligible intervals, "
          f"{nc_cnt:4d} NC intervals, {nc_sum:.2f} kW-excess total")

# Compare against stored result for v_threshold=258
stored_258 = aq(f"""
    SELECT
        SUM(nonconformance_sust_op_count) AS nc_count,
        SUM(nonconformance_sust_op_sum)   AS nc_sum,
        SUM(total_count)                  AS total_count
    FROM review_conformance_sust_op
    WHERE site_id = {SITE_ID} AND year = {YEAR} AND month = {MONTH}
      AND v_threshold = 258
""")
print(f"\nStored (review_conformance_sust_op, v=258):")
display(stored_258)


In [ ]:
# ── Plot: voltage + rolling average + generating flag for the site-month ───────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_plot = df.copy()
# Convert to AEST (fixed UTC+10) for display only — no DST
df_plot["t_local"] = df_plot["t_stamp"].dt.tz_convert("Etc/GMT-10")
df_plot["t_plot"]  = df_plot["t_local"].dt.tz_localize(None)   # strip tz for matplotlib

fig, axes = plt.subplots(3, 1, figsize=(14, 8), dpi=120, sharex=True)
fig.suptitle(
    f"Site {SITE_ID}  ·  {YEAR}-{MONTH:02d}  ·  Sustained-Op / Anti-Islanding verification",
    fontsize=10, fontweight="bold",
)

# Panel 1: Voltage + thresholds
ax = axes[0]
ax.plot(df_plot["t_plot"], df_plot["V"],         lw=1.0, color="#7c3aed", label="V (max, 5-min)")
ax.plot(df_plot["t_plot"], df_plot["V_rolling"], lw=1.5, color="#1565c0", ls="--",
        label="V rolling avg (≈10-min)")
for thr, lbl, col in [
    (258, "258 V (sust-op)", "#e65100"),
    (265, "265 V (AI OV1)",  "#c62828"),
    (275, "275 V (AI OV2)",  "#880e4f"),
]:
    ax.axhline(thr, color=col, lw=0.9, ls=":", alpha=0.8, label=lbl)
ax.set_ylabel("Voltage (V)", fontsize=8)
ax.legend(fontsize=7, ncol=3, loc="upper left")
ax.set_ylim(200, 300)
ax.grid(True, color="#ebebeb", lw=0.5)

# Panel 2: Active power + 4%-floor
ax = axes[1]
ax.plot(df_plot["t_plot"], df_plot["P_kW"], lw=1.0, color="#2e7d32", label="P (kW)")
floor = AS4777["TOL_FRAC"] * df_plot["S_rated"]
ax.plot(df_plot["t_plot"], floor, lw=0.8, color="#b45309", ls="--", label="4% nameplate floor")
ax.set_ylabel("Active power (kW)", fontsize=8)
ax.legend(fontsize=7, loc="upper left")
ax.grid(True, color="#ebebeb", lw=0.5)

# Panel 3: NC kW (sustained-op 258 V)
eligible_258 = (df_plot["V_rolling"] > 258) & (
    df_plot["generating"] | df_plot["prev_generating"]
)
df_plot["nc_258"] = np.where(
    eligible_258,
    np.maximum(0, df_plot["P_kW"] - AS4777["TOL_FRAC"] * df_plot["S_rated"]),
    0,
)
ax = axes[2]
ax.bar(df_plot["t_plot"], df_plot["nc_258"],
       width=pd.Timedelta(minutes=4.5), color="#c62828", alpha=0.8, linewidth=0)
ax.set_ylabel("NC excess (kW)
@ 258 V rolling", fontsize=8)
ax.set_xlabel("Time (AEST)", fontsize=8)
ax.grid(True, color="#ebebeb", lw=0.5, axis="y")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))

plt.tight_layout()
plt.show()


## 8. Fleet-level quick summary

Roll the review results up to the same format as the analysis notebook expects.
This is the same `site_conformance()` pattern from `02_conformance_curtailment_analysis.ipynb`.


In [ ]:
def site_conformance_review(table, nc_count_col, v_thr,
                             day_only=True,
                             thresh=AS4777["SITE_CONF_THRESH"]):
    """
    Roll daily×site grain up to one row per site, compute non-conforming fraction,
    and report fleet conformance rate.

    Parameters
    ----------
    table       : str   — Iceberg table name
    nc_count_col: str   — name of the interval-count NC column
    v_thr       : float — which threshold row to select
    day_only    : bool  — if True, restrict to day_night='day'
                          (use True for sustained-op; False for anti-islanding)
    thresh      : float — site conformant if NC fraction <= thresh (default 10%)
    """
    conds = [f"v_threshold = {v_thr}"]
    if day_only:
        conds.append("day_night = 'day'")
    where = "WHERE " + " AND ".join(conds)

    df = aq(f"""
        SELECT site_id,
               SUM({nc_count_col}) AS nonconf_count,
               SUM(total_count)    AS total_count
        FROM {table}
        {where}
        GROUP BY site_id
    """)
    df = df[df["total_count"] > 0].copy()
    df["nonconf_frac"] = df["nonconf_count"] / df["total_count"]
    df["conformant"]   = df["nonconf_frac"] <= thresh

    rate = df["conformant"].mean()
    nc_rate = 1 - rate
    print(
        f"{table} @ {v_thr}V  "
        f"| sites={len(df):,}  "
        f"| conformant={rate*100:.1f}%  "
        f"| non-conformant={nc_rate*100:.1f}%"
    )
    return df

sustop_sites  = site_conformance_review(
    "review_conformance_sust_op",
    "nonconformance_sust_op_count",
    v_thr=258, day_only=True,
)

ai_sites_265 = site_conformance_review(
    "review_conformance_antiisland",
    "nonconformance_antiisland_count",
    v_thr=265, day_only=False,   # keep night for anti-islanding
)

ai_sites_275 = site_conformance_review(
    "review_conformance_antiisland",
    "nonconformance_antiisland_count",
    v_thr=275, day_only=False,
)


In [ ]:
# ── Distribution of non-conforming fraction across sites ──────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), dpi=120)
thresh = AS4777["SITE_CONF_THRESH"]

specs = [
    (sustop_sites,  "Sustained-op (258V rolling avg, day)",     "#1565c0"),
    (ai_sites_265,  "Anti-islanding OV1 (265V rolling avg)",    "#c62828"),
    (ai_sites_275,  "Anti-islanding OV2 (275V rolling avg)",    "#880e4f"),
]

for ax, (d, title, colour) in zip(axes, specs):
    ax.hist(
        d["nonconf_frac"].clip(0, 1) * 100,
        bins=40, color=colour, alpha=0.80, edgecolor="k", linewidth=0.3,
    )
    ax.axvline(thresh * 100, color="red", ls="--", lw=1.2,
               label=f"{thresh*100:.0f}% threshold")
    ax.set_title(title, fontsize=8.5, fontweight="bold")
    ax.set_xlabel("% of evaluated intervals non-conforming", fontsize=8)
    ax.set_ylabel("sites", fontsize=8)
    ax.legend(fontsize=7.5)
    ax.tick_params(labelsize=8)

plt.suptitle("Non-conforming fraction per site (review tables)", fontsize=10,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 9. Open items / decisions before promoting to production

- [ ] **Confirm canonical `v_threshold`** for reporting:
  - Sustained-op: 258 V is unambiguous from clause 3.3.3.
  - Anti-islanding: 265 V (OV1) is the primary level; 275 V (OV2) is the backup.
  - Decide whether to report one level or both.
- [ ] **`day_only` for anti-islanding:** the inherited table kept night intervals.
  We default to `day_only=False` in `site_conformance_review` for anti-islanding
  (night OV events are unlikely but valid). Confirm for the report.
- [ ] **Cross-check site counts** with the inherited `conformance_sust_op_3w` and
  `conformance_antiisland` (cell 6, comparison block).
- [ ] **Sign off on resolution limitation note** for anti-islanding (cell 5 box).
- [ ] **Promote**: once satisfied, rename or `INSERT INTO` the production tables, or
  update the analysis notebook to point at `review_*` tables directly.
